# Notebook 05 — Machine Learning on Market Data

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Notebook 04 applied machine learning to a problem where the signal was real.
This notebook applies the *same machinery* to market prediction, where it
mostly is not — and that contrast is the lesson.

We do two things:

- **Task A: predict next month's direction.** This is what everyone tries
  first. We will do it carefully and watch it fail, and we will learn how to
  tell a failure from a success, which is harder than it sounds.
- **Task B: predict next month's volatility.** Same features, same models,
  different target — and here there is a real, if modest, edge. Understanding
  *why* the two differ is the point of the notebook.

> **Prerequisite:** run `03_features_and_targets.ipynb` first. It writes the
> file this notebook reads.

### How to use this notebook

Every cell is complete and runs as written. Run them top to bottom
(**Run All** works), read the explanation above each cell before you run it,
and compare what you see with what the text says you should see. The three
exercises at the end of each section come with worked solutions and a short
discussion that uses the numbers the code prints. When you have finished,
take the Canvas comprehension quiz for this notebook.

The modelling tools here — logistic regression, random forests, `Pipeline`,
`StandardScaler`, ROC-AUC, permutation importance — were introduced in depth
in Notebook 04. This notebook gives each a short recap and spends its
background sections on what is new: walk-forward validation, regression
models and their error metrics, volatility forecasting, and the economics of
why one target is predictable and the other is not.

## 0. Setup

The first cell loads the four libraries the notebook leans on and finds the
repository folder so file paths work on any machine.

- **`pathlib.Path`** (standard library) represents a file path as an object.
  The `/` operator joins path pieces, so `REPO / "data"` works the same on
  Windows and macOS.
- **NumPy** is the array-maths engine underneath almost all scientific Python.
  It stores numbers in compact typed arrays and does arithmetic on whole
  arrays at C speed. Here it supplies `np.sqrt`, `np.sort`, `np.full` and
  friends.
- **pandas** is the industry's standard table library: a `DataFrame` is a
  spreadsheet-like table with labelled rows and columns, built on NumPy. Banks,
  asset managers and fintechs use it for everything from trade blotters to
  risk reports.
- **matplotlib** is the base plotting library; `plt.subplots` creates a figure
  and axes, and every later chart is drawn onto those axes.

`RANDOM_STATE = 42` is a seed. Random forests draw random samples; fixing the
seed makes every run produce the same numbers, so your output matches the
text.

In [ ]:
# Path turns a file-system path into an object; "/" then joins folders portably on any OS
from pathlib import Path

# NumPy: fast numerical arrays; "np" is the universal nickname
import numpy as np
# pandas: labelled tables (DataFrames); "pd" is the universal nickname
import pandas as pd
# pyplot is matplotlib's charting interface; "plt" is the usual nickname
import matplotlib.pyplot as plt


# Define a helper that walks up from the current folder until it finds the repository root
def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""  # docstring: what the function does
    # Start from the given folder, or the current working directory; resolve() makes it absolute
    here = Path(start or Path.cwd()).resolve()
    # Check this folder first, then each parent folder in turn, moving towards the drive root
    for candidate in [here, *here.parents]:
        # The root is recognised by the price file it contains
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Found it: hand back that folder and stop searching
            return candidate
    # Ran out of parents without finding it: stop with a message that tells the student how to fix it
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # first half of the message
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # adjacent strings are joined automatically
    )


# Locate the repository root once and keep it for every later file path
REPO = find_repo_root()
# The data folder inside the repository
DATA = REPO / "data"
# Apply the course's shared chart style (fonts, colours, grid) from the .mplstyle file
plt.style.use(REPO / "fin4600.mplstyle")

# Show up to 40 columns when a DataFrame is displayed, instead of hiding the middle ones
pd.set_option("display.max_columns", 40)
# Allow 150 characters per printed line so wide tables wrap less
pd.set_option("display.width", 150)

# Fixed random seed, passed to every model that uses randomness, so results are reproducible
RANDOM_STATE = 42

### Loading the feature table

The next cell reads `monthly_features.csv`, the file Notebook 03 built. Each
row is one **stock-month**: one of 30 S&P 500 stocks at the end of one month.
The columns fall into three groups.

- **Identifiers:** `ticker`, `month`, `date`, `sector`, `close`.
- **Features**, all known at the month-end close, so they can legitimately be
  used to forecast:
  - `momentum_1m` … `momentum_12m` — the stock's trailing 1, 3, 6 and 12-month
    returns (`momentum_1m` is simply this month's return);
  - `volatility_21d`, `volatility_63d` — annualised standard deviation of
    daily returns over the last 21 and 63 trading days (about one and three
    months), scaled by $\sqrt{252}$;
  - `px_vs_200d_ma`, `px_vs_52w_high` — price relative to its 200-day moving
    average and to its 52-week high;
  - `volume_zscore` — how unusual this month's trading volume is, in
    standard deviations;
  - `vix_close`, `vix_change_5d`, `vix_vs_63d_avg` — the VIX "fear index"
    level, its 5-day change, and its level relative to its own 63-day
    average. These are market-wide, so all 30 stocks share the same values in
    a given month.
- **Targets**, which describe the *following* month and are therefore what we
  try to predict: `next_month_return`, `next_month_up` (1 if that return is
  positive) and `next_month_volatility` (annualised realised volatility next
  month).

Notebook 03 used `shift()` to line each month's features up with the next
month's outcome, which is what keeps look-ahead bias out of everything below.

Two new pandas tools appear. `pd.read_csv(..., parse_dates=["date"])` turns
the text dates into real `datetime64` values. `pd.PeriodIndex(..., freq="M")`
converts the `"2014-02"` strings into monthly **periods** — a value that means
"the whole of February 2014" rather than a single instant — which sort
correctly and make month arithmetic easy.

**What to look for:** 1,410 rows = 30 tickers × 47 months, running from
2014-02 to 2017-12, then the first three rows of the table.

In [ ]:
# Build the path to the processed feature file that Notebook 03 wrote
features_path = DATA / "processed" / "monthly_features.csv"

# Guard clause: if Notebook 03 has not been run, stop with a helpful explanation instead of a cryptic error
if not features_path.exists():
    # FileNotFoundError is the standard Python error for a missing file
    raise FileNotFoundError(
        "monthly_features.csv is missing.\n"  # \n starts a new line in the message
        "Run 03_features_and_targets.ipynb from top to bottom first — its last "  # adjacent strings join into one
        "cell writes the file this notebook needs."  # end of the message
    )

# Read the CSV into a DataFrame; parse_dates turns the "date" text column into datetime64 values
data = pd.read_csv(features_path, parse_dates=["date"])
# Replace the "YYYY-MM" text in "month" with monthly Period values, which sort and compare as months
data["month"] = pd.PeriodIndex(data["month"], freq="M")
# Sort by month, then ticker within each month, and renumber the rows 0..N-1 (drop=True discards the old index)
data = data.sort_values(["month", "ticker"]).reset_index(drop=True)

# The 12 input columns every model below will use, all known at month end
FEATURES = [
    "momentum_1m", "momentum_3m", "momentum_6m", "momentum_12m",  # trailing returns
    "volatility_21d", "volatility_63d",  # trailing realised volatility, annualised
    "px_vs_200d_ma", "px_vs_52w_high", "volume_zscore",  # trend, drawdown-from-high and volume signals
    "vix_close", "vix_change_5d", "vix_vs_63d_avg",  # market-wide fear gauge, the same for every stock in a month
]

# Summarise the table: total rows (with a thousands comma), distinct tickers and distinct months
print(f"{len(data):,} rows, {data['ticker'].nunique()} tickers, "
      f"{data['month'].nunique()} months")  # second half of the same f-string message
# First and last month in the sample
print(f"{data['month'].min()} to {data['month'].max()}")
# Display the first three rows to see the layout
data.head(3)

## 1. Task A — predicting direction

Target: `next_month_up`, 1 if next month's return is positive.

The benchmark to beat is the **base rate**: always predicting "up". Anything
that does not beat it is worse than useless, because it costs money to run.

#### Background: the classification baseline

Before judging any classifier, ask what a model with **no information** would
score. There are two natural "know-nothing" benchmarks:

- **The coin flip.** Guess up or down at random, 50/50. Expected accuracy is
  50% whatever the data, and its ROC-AUC is exactly 0.5 because its scores
  carry no ranking information.
- **The majority class.** Always predict the more common outcome. Its accuracy
  equals the **base rate** — the share of rows in that class. Stocks rise more
  often than they fall (they earn a risk premium on average), so here the
  majority class is "up" and the base rate is well above 50%.

The majority-class rule is the tougher benchmark and the right one. It uses no
features, costs nothing to run, and still beats the coin flip. A model that
merely matches it has learned the base rate and nothing else. The base rate is
just the mean of the 0/1 target: $\bar{y} = \frac{1}{n}\sum_i y_i$.

**What to look for:** the base rate printed as a percentage — the accuracy a
model must beat to be worth anything.

In [ ]:
# The mean of a 0/1 column is the share of 1s: the fraction of stock-months followed by an up month
base_rate = data["next_month_up"].mean()
# Format as a percentage with one decimal place (:.1%)
print(f"Base rate — share of months that were up: {base_rate:.1%}")
# Always predicting "up" is right exactly as often as the market actually goes up
print(f"So 'always predict up' scores {base_rate:.1%} accuracy.")

### Walk-forward validation

A single train/test split wastes data and gives one noisy number.
`TimeSeriesSplit` does better: it trains on an expanding window of history and
tests on the block that follows, repeatedly. Every test block is strictly in
the future relative to its training data.

This is the honest analogue of cross-validation for time series, and it is
what a research group would actually use.

#### Background: `TimeSeriesSplit` and walk-forward validation

**Intuition.** Ordinary k-fold cross-validation (Notebook 04) shuffles rows
into folds, so a model can be trained on 2017 and tested on 2015. For credit
applications that is fine — one borrower does not depend on the date of
another. For market data it is a leak: the model "knows the future" while
being tested on the past. Walk-forward validation mimics what a trader really
does: fit on everything up to today, forecast the next period, move forward,
refit, repeat.

**Algorithm.** `TimeSeriesSplit(n_splits=k)` cuts $N$ ordered observations
into $k+1$ consecutive blocks. Each of the last $k$ blocks holds
$\lfloor N/(k+1) \rfloor$ observations, and any remainder goes to the first
block. For fold $j = 1,\dots,k$:

- **train** on blocks $1,\dots,j$ (an *expanding* window — it grows each fold);
- **test** on block $j+1$, the block immediately after.

Every test block is strictly later than all of its training data, and the
test blocks never overlap, so together they give $k$ honest out-of-sample
scores rather than one.

**Hyper-parameter used here: `n_splits=5`.** With 47 months, each test block
is $\lfloor 47/6 \rfloor = 7$ months, and the 12 months left over form the
first training window: fold 1 trains on 12 months, fold 5 on 40. More splits
mean more test scores but smaller early training sets.

**One design choice worth noticing.** We split the array of **months**, not
the rows of `data`. Each month contains 30 stocks whose returns move together.
Splitting by month guarantees that all 30 stocks in a month land on the same
side of the split, so the model can never learn "what the market did in
March 2016" from 29 stocks and be tested on the 30th.

**Strengths:** honest, uses the data repeatedly, shows how performance
changes through time (regimes). **Weaknesses:** early folds have little
training data; results depend on which periods happen to be in the test
blocks. **In finance:** walk-forward backtesting is the standard for trading
signals, risk-model validation and any forecast that will be used through
time. Regulators' model-risk guidance (for example the Fed's SR 11-7) expects
out-of-time testing of this kind.

**What to look for:** five folds, each test block 7 months long and starting
the month after its training window ends, and training windows that start at
12 months and grow by 7 months each time.

In [ ]:
# TimeSeriesSplit generates forward-only train/test index pairs for ordered data
from sklearn.model_selection import TimeSeriesSplit

# The 47 distinct months in chronological order; we split months, not individual rows
months = np.sort(data["month"].unique())
# Five forward-walking folds over those months (7-month test blocks; the first training window is 12 months)
splitter = TimeSeriesSplit(n_splits=5)

# Header line; \n adds a blank line after it
print("Fold structure (by month, never overlapping, always forward):\n")
# split() yields (train positions, test positions); enumerate(..., start=1) numbers the folds from 1
for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
    # Convert positions into the actual months they refer to
    train_months, test_months = months[train_index], months[test_index]
    # Print first and last training month, how many there are, then the same for the test block
    print(f"  Fold {fold}:  train {train_months[0]} to {train_months[-1]}"
          f"  ({len(train_months):>2} months)"  # :>2 right-aligns the count in two characters
          f"   ->  test {test_months[0]} to {test_months[-1]}"  # first and last test month
          f"  ({len(test_months)} months)")  # size of the test block

### Running three classifiers through the walk-forward folds

The next cell is the core experiment of Task A. For each of three candidates
and each of the five folds it trains on the fold's past, predicts its future,
and records how well it did.

**Recap of the models (full treatment in Notebook 04):**

- **Logistic regression** turns a weighted sum of the features into a
  probability with the sigmoid, $P(\text{up}) = 1/(1+e^{-(\beta_0 + \beta^\top x)})$,
  choosing the $\beta$s by maximum likelihood. sklearn applies a mild L2
  penalty by default (`C=1.0`). `max_iter=2000` just gives the solver enough
  iterations to converge.
- **Random forest** averages many decision trees, each grown on a bootstrap
  sample of the rows and considering a random subset of features at each
  split. `n_estimators=300` trees; `min_samples_leaf=20` stops any leaf from
  holding fewer than 20 stock-months, which limits how finely a tree can carve
  up the training data (Exercise 1 relaxes this). `n_jobs=-1` uses every CPU
  core.
- **`StandardScaler` inside a `Pipeline`.** The scaler converts each feature to
  a z-score, $(x-\mu)/\sigma$, using the **training** mean and standard
  deviation only. Wrapping scaler and model in a `Pipeline` means `.fit` learns
  both from the training rows and `.predict` applies both to the test rows, so
  no test-period information leaks into the scaling.
- **"Always predict up"** is the majority-class baseline, represented by
  `None` so the loop knows not to fit anything.

**Recap of the metrics.** `accuracy_score` is the share of correct 0/1 calls.
`roc_auc_score` takes the predicted *probabilities* and measures ranking
skill: the probability that a randomly chosen up-month gets a higher score
than a randomly chosen down-month. 0.5 is a coin flip, 1.0 is perfect, and —
crucially — AUC does not reward a model for simply leaning towards the
majority class. `predict_proba(...)[:, 1]` returns the model's probability of
class 1 ("up"); thresholding at 0.5 turns it into a 0/1 call.

The loop also saves every out-of-sample probability in `predictions`, which
the backtest later uses.

**What to look for:** in the summary table, compare each model's
`mean_accuracy` with the "Always predict up" row, read `mean_auc` against
0.5, and look at `share_predicted_up`.

In [ ]:
# Random forest classifier: an average of many decision trees
from sklearn.ensemble import RandomForestClassifier
# Logistic regression: a linear model of the log-odds of "up"
from sklearn.linear_model import LogisticRegression
# accuracy_score = share of correct calls; roc_auc_score = ranking skill from probabilities
from sklearn.metrics import accuracy_score, roc_auc_score
# Pipeline chains preprocessing and model so both are fitted on training rows only
from sklearn.pipeline import Pipeline
# StandardScaler rescales each feature to mean 0, standard deviation 1
from sklearn.preprocessing import StandardScaler

# The three candidates, keyed by display name; None marks the no-model baseline
classifiers = {
    "Always predict up": None,  # majority-class baseline: nothing to fit
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),  # extra iterations so the solver converges
    "Random forest": RandomForestClassifier(  # ensemble of decision trees
        n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1  # 300 trees, leaves of at least 20 rows, all cores
    ),
}

# One dict of scores per (model, fold) will be appended here
rows = []
# Out-of-sample probabilities for every test row, kept for the backtest later
predictions = []

# Outer loop: each candidate model in turn; items() gives (name, estimator) pairs
for name, estimator in classifiers.items():
    # Inner loop: the same five walk-forward folds for every model, so the comparison is fair
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        # isin() keeps the rows whose month is in this fold's training months (all 30 stocks per month)
        train = data[data["month"].isin(months[train_index])]
        # The following 7 months form the test block
        test = data[data["month"].isin(months[test_index])]

        # The true 0/1 outcomes for the test block, as a NumPy array
        y_test = test["next_month_up"].to_numpy()

        # Baseline branch: no model at all
        if estimator is None:
            # ones_like makes an array of 1s the same shape as y_test: "up" for every row
            y_hat = np.ones_like(y_test)
            # A constant prediction has no ranking, so AUC is undefined; NaN means "not a number"
            auc = np.nan
        # Model branch: fit and predict
        else:
            # Build scaler + model, then fit both on the training rows only (12 features -> 0/1 target)
            pipeline = Pipeline(
                [("scale", StandardScaler()), ("model", estimator)]  # each step is a (name, object) pair
            ).fit(train[FEATURES], train["next_month_up"])  # fit returns the fitted pipeline itself
            # predict_proba returns an (n_rows, 2) array of P(down), P(up); [:, 1] keeps the P(up) column
            probability = pipeline.predict_proba(test[FEATURES])[:, 1]
            # Call "up" when the probability is at least 0.5; astype(int) turns True/False into 1/0
            y_hat = (probability >= 0.5).astype(int)
            # Ranking skill on the test block, computed from the probabilities, not the 0/1 calls
            auc = roc_auc_score(y_test, probability)
            # Keep identifiers, outcomes and the model's probability for every test row
            predictions.append(
                test[["month", "ticker", "next_month_return", "next_month_up"]]  # double brackets select several columns
                .assign(model=name, probability=probability, fold=fold)  # assign adds new columns and returns a copy
            )

        # Record this model's scores on this fold
        rows.append(
            {  # one record for this model and fold
                "model": name,  # which candidate
                "fold": fold,  # which fold (1-5)
                "accuracy": accuracy_score(y_test, y_hat),  # share of correct 0/1 calls
                "auc": auc,  # ranking skill (NaN for the baseline)
                "share_predicted_up": y_hat.mean(),  # mean of 0/1 calls = fraction of rows called "up"
            }
        )

# 15 rows (3 models x 5 folds) become a DataFrame
fold_results = pd.DataFrame(rows)
# Average over the folds for each model; named aggregation: new_column=(source_column, function)
summary = fold_results.groupby("model").agg(
    mean_accuracy=("accuracy", "mean"),  # average test accuracy across folds
    sd_accuracy=("accuracy", "std"),  # fold-to-fold spread: how noisy each estimate is
    mean_auc=("auc", "mean"),  # average AUC (NaN for the baseline)
    share_predicted_up=("share_predicted_up", "mean"),  # how often the model calls "up"
)
# Display rounded to three decimals
summary.round(3)

Read that table slowly, because it is the whole point of the notebook.

- Both models' mean accuracy is *below* "always predict up".
- AUC hovers near 0.5, which is the value of a coin flip; logistic
  regression's is actually below it.
- `share_predicted_up` shows what the models are actually doing. The random
  forest calls "up" about two thirds of the time, which partly reproduces the
  base rate; logistic regression calls "up" only about half the time, so it
  has not even learned that. In neither case does the leaning track the
  outcome, and neither model has found anything beyond the base rate.

A model that has learned only the base rate can still post a respectable
accuracy score while containing no information at all. That is why AUC, which
is insensitive to the base rate, is the number to read first.

The numbers you should see: "Always predict up" averages about 0.566 across
the five test blocks (a little below the full-sample 58.3%, because the test
blocks happen to contain slightly fewer up-months). Logistic regression scores
about 0.455 and the random forest about 0.513 — both *below* the do-nothing
rule — with mean AUCs of about 0.456 and 0.527. The forest's AUC is a whisker
above 0.5, but a difference that small is swamped by the fold-to-fold swings
(`sd_accuracy` is about 0.09 for both models).

### Picturing the folds

The next cell draws the per-fold accuracies as grouped bars, one group per
fold and one bar per model, with the overall base rate as a dashed line.
`fold_results.groupby("model")` yields the three models in alphabetical order
("Always predict up", "Logistic regression", "Random forest"), and `zip`
pairs each with a horizontal offset so the three bars sit side by side.
`ax.axhline` draws the horizontal reference line and `ax.annotate` labels it.

**What to look for:** no model's bar is consistently above the dashed line,
and the bars jump around from fold to fold by more than they differ from one
another. The random forest has one good fold (fold 2, about 0.64) and one very
poor one (fold 1, about 0.39): a single lucky block is exactly what noise
looks like.

In [ ]:
# One figure with one set of axes, 9.5 x 4.5 inches
fig, ax = plt.subplots(figsize=(9.5, 4.5))
# x-positions 0..4, one per fold
positions = np.arange(fold_results["fold"].nunique())
# Width of each bar; three bars fit in each group
width = 0.26

# Pair each model's rows with an offset (left, centre, right); groupby sorts the model names alphabetically
for offset, (name, group) in zip([-width, 0, width], fold_results.groupby("model")):
    # Draw this model's five fold accuracies (sorted by fold) as bars shifted by its offset
    ax.bar(positions + offset, group.sort_values("fold")["accuracy"],
           width=width, label=name)  # label feeds the legend

# Dashed horizontal line at the full-sample base rate
ax.axhline(base_rate, color="#52514e", linestyle="--", linewidth=1.2)
# Text label for that line, nudged 5 points above it at the left edge of the chart
ax.annotate(f"overall base rate {base_rate:.0%}", xy=(-0.45, base_rate),
            xytext=(0, 5), textcoords="offset points", ha="left",  # offset measured in points; left-aligned text
            fontsize=9, color="#52514e")  # small grey text
# Label the x-ticks "Fold 1" ... "Fold 5"
ax.set_xticks(positions, [f"Fold {f}" for f in sorted(fold_results['fold'].unique())])
# y-axis label
ax.set_ylabel("Accuracy")
# Chart title states the conclusion
ax.set_title("Direction prediction: neither model reliably clears the base rate")
# Fix the y-range so there is room for the legend at the top
ax.set_ylim(0, 0.95)
# Legend in one row of three, centred at the top of the axes
ax.legend(loc="upper center", ncols=3, fontsize=9, bbox_to_anchor=(0.5, 1.0))
# Hide vertical grid lines; they add clutter to a bar chart
ax.grid(axis="x", visible=False)
# Render the figure
plt.show()

### Is the difference statistically meaningful?

Suppose one model had come out two points above the base rate. Would that be
a discovery? Compute the standard error and see how wide the uncertainty is.

For a proportion, $\text{SE} = \sqrt{p(1-p)/n}$.

#### Background: how noisy is an accuracy score?

Each test prediction is right or wrong, so an accuracy score is a sample
proportion. If the true hit rate is $p$ and we score $n$ independent
predictions, the number correct is binomial and the estimate $\hat{p}$ has
standard error $\sqrt{p(1-p)/n}$. By the central limit theorem $\hat{p}$ is
roughly normal, so a 95% interval is $\hat{p} \pm 1.96\,\text{SE}$.

The cell uses $n$ = 1,410 rows / 6 ≈ 235 stock-months as the size of a test
block, and $p$ = the base rate. Three caveats make the real uncertainty
*larger*: the actual test blocks are a little smaller (7 months × 30 stocks =
210 rows); the 30 stocks in a month move together, so the predictions are not
independent; and the base rate itself shifts from block to block.

**What to look for:** a standard error of about 0.032, i.e. a band roughly
six percentage points either side of the base rate (about 0.520 to 0.646).
Then compare the band with the summary table above. The cell checks each
model's mean accuracy against the band for you. "Always predict up" sits
inside it, by construction. The two models sit *below* the lower edge:
logistic regression (about 0.455) well below, the random forest (about 0.513)
just below. Nothing sits above it, and only an accuracy above the band would
be evidence of skill. The models call "down" without skill — logistic
regression about half the time, the forest about a third — so they do *worse*
than the do-nothing rule.

In [ ]:
# One sixth of all rows approximates one test block (the actual blocks are 7 of 47 months); int() drops any fraction
n_test = int(len(data) / 6)  # roughly the size of one test block
# Use the base rate as the hit rate a know-nothing rule would achieve
p = base_rate
# Binomial standard error of a proportion: sqrt(p(1-p)/n)
standard_error = np.sqrt(p * (1 - p) / n_test)

# Report the block size
print(f"Test block size: about {n_test} observations")
# Report the standard error to three decimals
print(f"Standard error of an accuracy estimate: {standard_error:.3f}")
# 95% normal interval: base rate plus or minus 1.96 standard errors
print(f"A 95% interval around the base rate runs "
      f"{p - 1.96 * standard_error:.3f} to {p + 1.96 * standard_error:.3f}")  # lower and upper bounds
# Blank line for readability
print()
# The interpretation, as the original author wrote it
print("Any accuracy inside that band is indistinguishable from guessing.")
# Edges of the 95% band, kept so each model can be checked against them
band_lower, band_upper = p - 1.96 * standard_error, p + 1.96 * standard_error
# Loop over the summary table from section 1: one (model name, mean accuracy) pair per row
for model_name, accuracy in summary["mean_accuracy"].items():
    # Classify the accuracy as below, inside or above the band
    where = "below" if accuracy < band_lower else ("above" if accuracy > band_upper else "inside")
    # Report it; :<22 left-aligns the name in a 22-character column so the numbers line up
    print(f"   {model_name:<22} {accuracy:.3f}  {where} the band")
# Only an accuracy ABOVE the band would be evidence of skill
if (summary["mean_accuracy"] > band_upper).any():
    # At least one model clears the band
    print("At least one model sits above the band: worth a closer look.")
else:  # no model clears the band
    # Nothing clears it, so nothing beats guessing "up"
    print("No model sits above the band, so none shows skill beyond guessing.")

### Does it make money? The only test that matters

Accuracy is not the objective. Build the portfolio the model implies — each
month, hold an equal-weight basket of the names it scores highest — and
compare it with simply holding all 30.

**How the backtest works.** The loop above saved every random-forest test
probability, so each month in the five test blocks (35 months, 2015-02 to
2017-12) has an out-of-sample score for all 30 stocks.

- `pd.concat` stacks the saved per-fold tables into one.
- `monthly_selection` takes one month's 30 rows, keeps the `k` = 10 with the
  highest probability (`nlargest`) and returns their average next-month return
  — the return of an equal-weighted basket.
- `groupby("month").apply(monthly_selection)` runs it for every month, giving
  the strategy's monthly return series; the benchmark is the plain average of
  all 30 stocks each month.
- `(1 + r).cumprod()` compounds monthly returns into the growth of \$1.
- **Annualising.** Annual return is $(\text{growth})^{12/T} - 1$ for $T$
  months. Annual volatility is the monthly standard deviation times
  $\sqrt{12}$, the monthly analogue of the $\sqrt{252}$ rule for daily data
  (variances of independent periods add, so standard deviations grow with the
  square root of time).
- **Return/vol** is a Sharpe-style ratio without the risk-free rate: return
  earned per unit of risk.

**What to look for:** the orange model line ending below the blue
hold-everything line, and in the table roughly 10.9% a year for the strategy
against 13.4% for equal weight, with the strategy also a little *more*
volatile (about 10.2% against 9.5%) — so a clearly lower return-to-volatility
ratio (about 1.07 versus 1.42). Before any trading costs.

In [ ]:
# Stack the per-fold prediction tables into one long DataFrame with a fresh 0..N-1 index
predicted = pd.concat(predictions, ignore_index=True)
# Keep only the random forest's predictions (boolean mask on the "model" column)
forest_predictions = predicted[predicted["model"] == "Random forest"]

# Each month, go long the top third by predicted probability
def monthly_selection(group, fraction=1 / 3):
    # Number of stocks to hold: a third of the month's names (10 of 30), but never fewer than one
    k = max(1, int(len(group) * fraction))
    # nlargest keeps the k rows with the highest probability; the mean of their returns is the equal-weight basket return
    return group.nlargest(k, "probability")["next_month_return"].mean()


# Apply the selection to each month separately -> a Series of monthly strategy returns indexed by month
strategy = forest_predictions.groupby("month").apply(
    monthly_selection, include_groups=False  # include_groups=False: don't pass the "month" column into the function
).rename("strategy")  # name the Series so it becomes a column label below
# Benchmark: the average return of all 30 stocks each month (equal weight, no model)
benchmark = forest_predictions.groupby("month")["next_month_return"].mean().rename("equal_weight")

# Put the two series side by side as columns (axis=1) and drop any month missing either
backtest = pd.concat([strategy, benchmark], axis=1).dropna()
# Convert the monthly Period index to Timestamps so matplotlib can plot dates
backtest.index = backtest.index.to_timestamp()

# Compound: cumulative product of (1 + monthly return) = growth of $1
cumulative = (1 + backtest).cumprod()

# New figure for the growth chart
fig, ax = plt.subplots(figsize=(9.5, 4.8))
# Benchmark line in blue
ax.plot(cumulative.index, cumulative["equal_weight"], label="Hold all 30 equally",
        color="#2a78d6")  # blue
# Strategy line in orange
ax.plot(cumulative.index, cumulative["strategy"], label="Model's top third each month",
        color="#eb6834")  # orange
# Reference line at $1, the starting value
ax.axhline(1.0, color="#52514e", linewidth=0.8, linestyle="--")
# Title reminds the reader that costs are not yet included
ax.set_title("Out-of-sample backtest, before any trading costs")
# y-axis label
ax.set_ylabel("Growth of $1")
# No x-axis label needed; the dates speak for themselves
ax.set_xlabel("")
# Legend in the upper-left corner
ax.legend(loc="upper left")
# Render the figure
plt.show()

# Annualised return: final growth raised to 12/T (T = number of months), minus 1
annualised = (cumulative.iloc[-1] ** (12 / len(backtest)) - 1)
# Annualised volatility: monthly standard deviation times sqrt(12)
volatility = backtest.std() * np.sqrt(12)
# Header row; :<28 left-aligns in 28 characters, :>14 and :>12 right-align
print(f"{'':<28}{'Ann. return':>14}{'Ann. vol':>12}{'Return/vol':>12}")
# One printed row per column of the backtest (strategy, then equal_weight)
for column in backtest.columns:
    # Name, annual return and volatility as percentages, then their ratio
    print(f"{column:<28}{annualised[column]:>13.1%}{volatility[column]:>12.1%}"
          f"{annualised[column] / volatility[column]:>12.2f}")  # return per unit of risk

# Reminder of the costs a real strategy would pay
print("\nAnd this ignores commissions, bid-ask spread, and market impact.")
# Monthly rebalancing means paying them every month
print("A monthly-rebalanced strategy would pay all three, twelve times a year.")

### Why this is the expected result, not a bug

Prices already reflect what is publicly known. A pattern visible in five
years of daily closes, using features any undergraduate can compute, has been
arbitraged away by people with better data, faster systems, and more capital.

What firms that *do* extract signal from markets actually have:

- data nobody else has (order flow, satellite imagery, card transactions)
- a speed advantage measured in microseconds
- a structural role that earns a spread rather than a forecast
- a genuine risk premium they are paid to bear

None of those is a random forest on public closing prices.

**The professionally valuable skill here is recognising a null result and
saying so.** A student who reports "I tried and it does not work, here is the
evidence" has learned more than one who tortures the data until it confesses.

#### Background: why price direction is (nearly) unpredictable

The argument above is the **efficient markets hypothesis** (Fama, 1970),
and it is worth seeing why it bites so hard on *direction* specifically.

- **The arbitrage argument.** Suppose a public signal reliably said "this stock
  will rise next month". Traders would buy it *today*, pushing today's price
  up until the expected extra return is gone. A predictable price move is a
  profit opportunity, and profit opportunities are competed away. What is
  left over is, to a first approximation, unpredictable: tomorrow's price
  equals today's plus a fair return for risk plus noise. That is the
  **random walk** picture of prices.
- **The forms of efficiency.** *Weak form*: past prices and volumes cannot
  predict future returns — exactly the features this notebook uses.
  *Semi-strong form*: no public information helps. *Strong form*: not even
  private information helps (rarely believed; insider-trading law exists
  because it is false).
- **Signal-to-noise.** Even where small predictable components survive —
  academic momentum and value premia are real but modest — they explain a
  tiny share of the month-to-month variance of an individual stock. A feature
  with a genuine correlation of 0.05 with next month's return would be an
  economically interesting finding, yet it explains a quarter of one per cent
  of the variance ($0.05^2$) and yields an AUC barely above 0.5 on a few
  hundred observations.
- **The Grossman–Stiglitz paradox.** If markets were *perfectly* efficient,
  nobody would pay to gather information, so prices could not become
  efficient. The resolution is that markets are efficient enough that only
  those with an edge in data, speed or cost earn a return for the effort —
  the list above.

None of this applies to volatility in the same way, and Task B will show why.

### Exercise 1 (worked solution)
Rerun the fold loop with `min_samples_leaf=1` on the random forest — a model
free to memorise. Training accuracy will go up sharply. What happens to test
accuracy, and what is that phenomenon called?

#### Background: overfitting and the generalisation gap

A model **overfits** when it learns the quirks of its particular training
sample — noise — as if they were signal. The symptom is a gap between
**training accuracy** (scored on the rows it learned from) and **test
accuracy** (scored on new rows). A decision tree with no limit on leaf size
can keep splitting until every leaf holds one training row, which classifies
every training row perfectly: it has memorised the sample. `min_samples_leaf`
is the brake: with 20, each leaf must average at least 20 stock-months, which
blurs out row-specific noise. A random forest averages many such trees, which
reduces variance, but cannot rescue a target that contains no signal.

**The solution below** reruns the walk-forward loop for the random forest
twice — once with the original `min_samples_leaf=20` and once with
`min_samples_leaf=1` — and records, for each fold, accuracy on the training
rows as well as on the test rows. It uses new variable names (`overfit_rows`,
`train_ex1`, …) so nothing the later cells depend on is overwritten.

In [ ]:
# Compare two random forests that differ only in min_samples_leaf: 20 (the setting used above) and 1
leaf_settings = [20, 1]
# One row per (setting, fold) will be collected here; a new name so `rows` above is untouched
overfit_rows = []

# Outer loop: try each leaf setting in turn
for leaf in leaf_settings:
    # Inner loop: the same five walk-forward folds as the main experiment
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        # Training rows: every stock-month whose month falls in this fold's training window
        train_ex1 = data[data["month"].isin(months[train_index])]
        # Test rows: the eight months that follow, never seen during fitting
        test_ex1 = data[data["month"].isin(months[test_index])]
        # Same pipeline as before; only min_samples_leaf changes between the two settings
        forest_ex1 = Pipeline(
            [  # the two pipeline steps, in order
                ("scale", StandardScaler()),  # z-score each feature using training-fold statistics only
                ("model", RandomForestClassifier(  # the forest whose leaf size we vary
                    n_estimators=300,  # 300 trees, as in the main loop
                    min_samples_leaf=leaf,  # 20 = cautious leaves; 1 = a leaf may hold a single stock-month
                    random_state=RANDOM_STATE,  # fixed seed so the result is reproducible
                    n_jobs=-1,  # use every CPU core
                )),
            ]
        ).fit(train_ex1[FEATURES], train_ex1["next_month_up"])  # fit scaler and forest on the training rows only
        # Record accuracy on the rows the model trained on AND on the unseen test block
        overfit_rows.append(
            {  # one record per setting and fold
                "min_samples_leaf": leaf,  # which setting this row belongs to
                "fold": fold,  # which walk-forward fold
                # predict() returns 0/1 labels; accuracy_score is the share that match the truth
                "train_accuracy": accuracy_score(
                    train_ex1["next_month_up"], forest_ex1.predict(train_ex1[FEATURES])  # truth vs predictions on rows it has seen
                ),
                # Accuracy on the unseen test block: the number that matters
                "test_accuracy": accuracy_score(
                    test_ex1["next_month_up"], forest_ex1.predict(test_ex1[FEATURES])  # truth vs predictions on new rows
                ),
                # AUC on the test block: 0.5 is a coin flip, whatever the accuracy says
                "test_auc": roc_auc_score(
                    test_ex1["next_month_up"], forest_ex1.predict_proba(test_ex1[FEATURES])[:, 1]  # P(up) for each test row
                ),
            }
        )

# Turn the list of dicts into a DataFrame: one row per setting and fold
overfit_results = pd.DataFrame(overfit_rows)
# Average the five folds for each setting; the gap column is training minus test accuracy
overfit_summary = overfit_results.groupby("min_samples_leaf").agg(
    train_accuracy=("train_accuracy", "mean"),  # how well it fits data it has already seen
    test_accuracy=("test_accuracy", "mean"),  # how well it does on the future
    test_auc=("test_auc", "mean"),  # ranking skill on the future
)
# Generalisation gap: a large positive number is the signature of overfitting
overfit_summary["gap"] = overfit_summary["train_accuracy"] - overfit_summary["test_accuracy"]
# Print the per-fold test accuracies side by side so the fold-to-fold noise is visible
print(overfit_results.pivot(index="fold", columns="min_samples_leaf", values="test_accuracy").round(3))
# Show the summary, rounded to three decimals
overfit_summary.round(3)

**Discussion.** With `min_samples_leaf=1` the forest scores a perfect
**1.00** training accuracy, up from about **0.78** with
`min_samples_leaf=20`. Test accuracy does not follow it: it is about
**0.504** against **0.513**, and test AUC is about **0.514** against
**0.527** — both indistinguishable from a coin flip and from each other
(per-fold test accuracy with leaf size 1 wanders from about 0.438 to 0.562).
The generalisation gap roughly doubles, from about 0.27 to 0.50.

The phenomenon is **overfitting**: the model has memorised the training
sample, noise included, and that memory is worthless on new data. Two further
points. First, even the "cautious" setting overfits — 78% in-sample against
51% out-of-sample — which is why a training-set score should never be
reported as evidence of skill. Second, test accuracy does not collapse here
because there was no real signal to lose; on a problem with genuine signal
(Notebook 04), overfitting usually shows up as test performance falling
while training performance rises.

## 2. Task B — predicting volatility

Same features. Same validation. Different target: next month's realised
volatility.

The economic reason to expect a different answer: **direction is competed
away, but risk is persistent.** If a stock has been volatile for a month, it
is likely to be volatile next month. That is not an arbitrage opportunity —
you cannot buy last month's volatility — so nobody trades it away.

#### Background: why volatility clusters

Look at any daily return chart and you see calm stretches and stormy
stretches: big moves (of either sign) tend to be followed by big moves, small
by small. This is **volatility clustering**, first documented by Mandelbrot
(1963), and it is one of the most robust facts in finance. Reasons:

- **News arrives in clusters.** Earnings seasons, central-bank cycles, crises
  and recessions produce bursts of information that take weeks to digest.
- **Leverage and feedback.** A falling price raises a firm's leverage, which
  makes its equity riskier; forced selling, margin calls and risk limits
  amplify moves once they start.
- **Regimes.** Markets alternate between low-uncertainty and high-uncertainty
  states (2017 versus August 2015 or March 2020), and these states last
  months.

So the *sign* of the return is close to unpredictable while its *size* is
quite predictable. Models built on this — Engle's ARCH (1982, Nobel prize)
and Bollerslev's GARCH, $\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta\sigma_{t-1}^2$
— are the industry workhorses for volatility forecasting.

**Realised volatility** here is the standard deviation of a stock's daily
returns over a month, annualised by multiplying by $\sqrt{252}$ (252 trading
days a year). The target `next_month_volatility` is that number for the coming
month; the feature `volatility_21d` is the same calculation for the month just
ended.

**What the next cell does.** It keeps the rows with a volatility target
(`dropna(subset=[...])` drops rows where that column is missing; `.copy()`
makes an independent table), draws a scatter of this month's against next
month's volatility with a 45-degree "no change" line, and prints their
correlation with `Series.corr` (Pearson correlation, from -1 to 1).

**What to look for:** a clear upward-sloping cloud around the diagonal, and a
correlation of about 0.411.

In [ ]:
# Keep rows that have a volatility target; .copy() makes an independent table rather than a view of data
volatility_data = data.dropna(subset=["next_month_volatility"]).copy()
# Row count, with a thousands comma
print(f"{len(volatility_data):,} rows with a volatility target")

# A roughly square figure suits a scatter of one volatility against another
fig, ax = plt.subplots(figsize=(6.5, 5.5))
# Each dot is one stock-month: x = this month's volatility, y = next month's
ax.scatter(volatility_data["volatility_21d"], volatility_data["next_month_volatility"],
           s=12, alpha=0.3, color="#2a78d6", edgecolor="none")  # small, semi-transparent dots so overlaps show density
# Axis limit at the 99.5th percentile of either column, so a few extreme points don't squash the chart
limit = volatility_data[["volatility_21d", "next_month_volatility"]].quantile(0.995).max()
# The 45-degree line: points on it had the same volatility in both months
ax.plot([0, limit], [0, limit], color="#52514e", linestyle="--", linewidth=1)
# Same range on both axes so the diagonal really is 45 degrees
ax.set_xlim(0, limit)
# y-range matches the x-range
ax.set_ylim(0, limit)
# x-axis label
ax.set_xlabel("This month's realised volatility")
# y-axis label
ax.set_ylabel("Next month's realised volatility")
# Title states the finding
ax.set_title("Volatility persists — the same chart for returns is a shapeless cloud")
# Turn the vertical grid lines on (the style sheet may switch them off)
ax.grid(axis="x")
# Render the figure
plt.show()

# Pearson correlation between this month's and next month's volatility
correlation = volatility_data["volatility_21d"].corr(volatility_data["next_month_volatility"])
# Print it to three decimals
print(f"Correlation between this month's and next month's volatility: {correlation:.3f}")

Compare that with the largest feature-to-return correlation from Notebook 03,
which was about 0.10 (`vix_vs_63d_avg`). This is about four times stronger,
and it is visible to the naked eye.

### The baseline that matters

A regression model fitted here can report a respectable *in-sample* $R^2$
and look impressive. Do not be impressed: as the next cell shows, the
*out-of-sample* $R^2$ is negative for every model. The honest benchmark is not "predict the average" — it is
**persistence**: predict that next month's volatility equals this month's.

That naive rule is free, requires no model, and is hard to beat. If the
machine learning model cannot beat it, the model has added nothing.

The two free baselines in the next cell are therefore:

- **Persistence (no model):** forecast = `volatility_21d`, this month's
  realised volatility.
- **Train-set mean (no model):** forecast = the average
  `next_month_volatility` in the training window, the same number for every
  stock — the regression analogue of "always predict up".

#### Background: linear regression and ridge regression

**Linear regression (ordinary least squares, OLS).** Predict the target as a
weighted sum of the features, $\hat{y} = \beta_0 + \beta_1 x_1 + \dots + \beta_{12} x_{12}$,
and choose the $\beta$s to minimise the sum of squared errors,
$\sum_i (y_i - \hat{y}_i)^2$. There is a closed-form answer,
$\hat\beta = (X^\top X)^{-1} X^\top y$, so fitting is instant. It is the same
tool as the CAPM beta regression, with 12 regressors instead of one.

*Weakness:* when features are strongly correlated with each other — the two
volatility windows, the four momentum horizons, the three VIX measures —
$X^\top X$ is nearly singular and OLS can assign huge positive and negative
coefficients that cancel on the training data but blow up on new data. With
only a few hundred training rows in the early folds, it fits noise.

**Ridge regression** adds an **L2 penalty** on the size of the coefficients:
$$\min_\beta \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j \beta_j^2 .$$
The hyper-parameter **`alpha`** sets the price of a large coefficient:
`alpha=0` is plain OLS; a large `alpha` shrinks every coefficient towards zero
and the forecast towards the average. `alpha=10.0` here is a moderate amount
of shrinkage. Correlated features end up sharing the weight instead of
fighting over it. Because the penalty depends on coefficient size, features
must be on comparable scales — one reason `StandardScaler` sits in front of
the model. (Logistic regression's `C` from Notebook 04 is the same idea,
expressed as $1/\alpha$.)

*Strengths:* fast, transparent coefficients, stable. *Weaknesses:* only
captures linear, additive relationships. *In finance:* ridge and its
relatives (lasso, elastic net) are standard in factor models, risk-model
estimation and return forecasting, wherever there are many correlated inputs.

#### Background: `RandomForestRegressor`

The same forest as in Task A, but each tree predicts a number instead of a
class. A regression tree splits the data on the feature and threshold that
most reduce the **squared error** within the two child groups (instead of Gini
impurity), and each leaf predicts the **average** target of its training rows.
The forest averages the predictions of 300 trees (`n_estimators=300`), each
grown on a bootstrap sample with a random subset of features at each split.
`min_samples_leaf=10` means every leaf averages at least 10 stock-months,
which smooths the forecast. Forests capture non-linearities and interactions
("high VIX *and* high own volatility") without being told about them, at the
cost of being harder to explain than a coefficient.

#### Background: MAE and R²

- **Mean absolute error**, $\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$,
  is the average size of a miss, in the target's own units — here annualised
  volatility points, so an MAE of 0.054 means forecasts are off by about 5.4
  volatility points on average. It is easy to explain and is not dominated by
  a handful of extreme months.
- **$R^2$**, $1 - \dfrac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$,
  compares the model's squared errors with those of predicting the mean
  $\bar{y}$. In-sample OLS $R^2$ lies between 0 and 1. **Out of sample it can
  be negative.** `r2_score` uses $\bar{y}$ = the mean of the *test* block —
  a number nobody knew in advance. If the volatility level shifts between
  training and test periods, even a sensible forecast can miss that level by
  more than the test block's own mean would, and $R^2$ goes below zero. A
  negative out-of-sample $R^2$ therefore means "worse than a benchmark that
  cheated by peeking at the test average", not necessarily "useless".

**What the next cell does.** It runs all five candidates through the same five
walk-forward folds. String flags (`"persistence"`, `"constant"`) mark the two
baselines; `np.full(n, value)` builds an array of $n$ copies of the training
mean. Real models go through the scaler pipeline as in Task A, and
`predict` returns numbers rather than probabilities. The `if len(test) == 0:
continue` line is a safety check that skips a fold with no rows.

**What to look for:** the table sorted best (lowest `mean_mae`) first: ridge
about 0.0538, random forest 0.0545, linear regression 0.0550, then the two
free baselines, persistence 0.0596 and train-set mean 0.0601. All three fitted
models beat both baselines on average, by about half a volatility point. Note
also linear regression's `worst_fold_mae` of 0.0750, the worst single fold of
any candidate, and a `mean_r2` column that is negative for every row.

In [ ]:
# Random forest for a numeric target: leaves predict averages instead of classes
from sklearn.ensemble import RandomForestRegressor
# Plain least-squares regression, and ridge (least squares with an L2 penalty on coefficient size)
from sklearn.linear_model import LinearRegression, Ridge
# MAE = average absolute miss; r2_score = 1 - SSE / SS around the test mean
from sklearn.metrics import mean_absolute_error, r2_score

# Five candidates; the two strings are flags for the no-model baselines handled inside the loop
regressors = {
    "Persistence (no model)": "persistence",  # forecast = this month's volatility
    "Train-set mean (no model)": "constant",  # forecast = training-window average for everyone
    "Linear regression": LinearRegression(),  # OLS on all 12 features
    "Ridge regression": Ridge(alpha=10.0),  # OLS plus an L2 penalty; alpha sets its strength
    "Random forest": RandomForestRegressor(  # ensemble of regression trees
        n_estimators=300, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1  # 300 trees, leaves of at least 10 rows
    ),
}

# Fresh list for this experiment's scores (reuses the name; Task A's table is already in fold_results)
rows = []
# Outer loop over the five candidates
for name, estimator in regressors.items():
    # Inner loop over the same walk-forward folds used in Task A
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        # Training window rows
        train = volatility_data[volatility_data["month"].isin(months[train_index])]
        # Following test block rows
        test = volatility_data[volatility_data["month"].isin(months[test_index])]
        # Safety check: skip a fold if the test block happens to be empty
        if len(test) == 0:
            # continue jumps straight to the next fold
            continue

        # True next-month volatility for the test rows
        y_test = test["next_month_volatility"].to_numpy()

        # Persistence baseline: next month's volatility = this month's
        if estimator == "persistence":
            # This month's 21-day realised volatility is the forecast
            y_hat = test["volatility_21d"].to_numpy()
        # Constant baseline: the training-window mean, repeated for every test row
        elif estimator == "constant":
            # np.full(n, v) makes an array of n copies of v
            y_hat = np.full(len(test), train["next_month_volatility"].mean())
        # A real model: scale, fit on training rows, predict the test rows
        else:
            # Same scaler-plus-model pipeline as Task A, now with a numeric target
            pipeline = Pipeline(
                [("scale", StandardScaler()), ("model", estimator)]  # (name, step) pairs
            ).fit(train[FEATURES], train["next_month_volatility"])  # learns scaling and coefficients/trees from training rows
            # predict returns one volatility forecast per test row
            y_hat = pipeline.predict(test[FEATURES])

        # Score this model on this fold
        rows.append(
            {  # one record for this model and fold
                "model": name,  # which candidate
                "fold": fold,  # which fold
                "r2": r2_score(y_test, y_hat),  # out-of-sample R squared (can be negative)
                "mae": mean_absolute_error(y_test, y_hat),  # average absolute miss in volatility points
            }
        )

# 25 rows (5 models x 5 folds) become a DataFrame
volatility_results = pd.DataFrame(rows)
# Average across folds per model, also keeping the worst fold, then sort best-first
volatility_summary = volatility_results.groupby("model").agg(
    mean_mae=("mae", "mean"),  # typical error
    worst_fold_mae=("mae", "max"),  # error in the model's worst period
    mean_r2=("r2", "mean"),  # average out-of-sample R squared
).sort_values("mean_mae")  # lowest error at the top
# Display rounded to four decimals
volatility_summary.round(4)

### Per-fold detail

Averages can hide a model that wins big once and loses the rest of the time.
The next cell reshapes the results with `pivot` — models as rows, folds as
columns, MAE in the cells — and reorders the rows to match the summary
(`.loc[volatility_summary.index]`). It then compares the random forest with
persistence fold by fold: `forest_mae < persistence_mae` gives five True/False
values, and `.sum()` counts the Trues.

**What to look for:** folds 1 and 2 (February 2015 to March 2016) have the
largest errors: fold 1 is the worst for every fitted model and fold 2 for both
baselines. In fold 1, trained on only 12 months, every fitted model loses to
both free baselines. The random forest beats persistence in 3 of 5 folds
(losing folds 1 and 4), and the mean MAE improvement over persistence is
about 8.6%. Ridge, for comparison, beats persistence in every fold but the
first.

In [ ]:
# Per-fold detail, because the averages hide a lot
# pivot reshapes long to wide: one row per model, one column per fold, MAE values in the cells
per_fold = volatility_results.pivot(index="model", columns="fold", values="mae")
# Reorder the rows to match the summary table's best-to-worst order
per_fold = per_fold.loc[volatility_summary.index]
# Table heading
print("Mean absolute error by fold (lower is better)\n")
# to_string() prints the whole table without truncation
print(per_fold.round(4).to_string())

# The random forest's five fold MAEs
forest_mae = per_fold.loc["Random forest"]
# Persistence's five fold MAEs
persistence_mae = per_fold.loc["Persistence (no model)"]
# Element-wise comparison gives True/False per fold; sum counts the Trues
wins = int((forest_mae < persistence_mae).sum())
# Report the win count
print(f"\nRandom forest beats persistence in {wins} of {len(forest_mae)} folds.")
# Relative improvement: 1 - (forest MAE / persistence MAE), as a percentage
print(f"Mean improvement in MAE: {1 - forest_mae.mean() / persistence_mae.mean():.1%}")

### Picturing the comparison

A horizontal bar chart of mean MAE, worst at the top and best at the bottom
(`barh` draws horizontal bars). The list comprehension colours the two
no-model baselines orange and the fitted models blue, and the final loop
writes each bar's value just past its end.

**What to look for:** all three blue fitted-model bars are shorter than both
orange baselines. Ridge is the shortest, with the random forest and plain
linear regression close behind; the three fitted models differ from each other
by much less than they differ from the baselines.

In [ ]:
# Reverse the sort so the worst model is first; barh draws the first row at the bottom
plot_order = volatility_summary.sort_values("mean_mae", ascending=False)

# New figure for the bar chart
fig, ax = plt.subplots(figsize=(9, 4))
# Orange for the no-model baselines, blue for fitted models (a list comprehension over the model names)
colors = ["#eb6834" if "no model" in name else "#2a78d6" for name in plot_order.index]
# One horizontal bar per model, length = mean MAE
ax.barh(plot_order.index, plot_order["mean_mae"], color=colors, height=0.6)
# Title says what is plotted and which direction is good
ax.set_title("Mean absolute error predicting next month's volatility — lower is better")
# x-axis label with units
ax.set_xlabel("Mean absolute error (annualised volatility points)")
# Show vertical grid lines to help read bar lengths
ax.grid(axis="x")
# Hide horizontal grid lines
ax.grid(axis="y", visible=False)
# enumerate gives the bar's position (y) and its value
for y, value in enumerate(plot_order["mean_mae"]):
    # Write the value just to the right of the bar's end
    ax.text(value + 0.0009, y, f"{value:.4f}", va="center", fontsize=9, color="#52514e")
# Leave 20% headroom on the right for the labels
ax.set_xlim(0, plot_order["mean_mae"].max() * 1.2)
# Render the figure
plt.show()

### What this table does and does not say

On average all three fitted models beat both free baselines, and the best of
them is the simplest regularised one: **ridge regression**, with a mean MAE of
about 0.0538, just ahead of the random forest at 0.0545. Ridge beats both
baselines in four folds out of five and cuts mean absolute error by about 10%
relative to persistence (0.0538 against 0.0596). The forest's edge over
persistence is about 9%, but it wins in only 3 of the 5 folds. That is a
**real but modest** edge, and it is worth stating in exactly those words
rather than dressing it up. Nor is it a win for elaborate machine learning: a
penalised linear model does as well as 300 trees.

Now look at the `mean_r2` column: it is negative for every model, the
baselines included. A negative out-of-sample $R^2$ means the model did worse
than predicting the test block's own average — which nobody could have known
in advance, so it is a brutally hard benchmark. On 47 months with big shifts
in the market's volatility level between folds, $R^2$ is unstable and mostly
measures those shifts. **Mean absolute error against a named baseline is the
more honest summary here**, and it tells a consistent story that $R^2$ does
not.

Two further things worth noticing:

- **Plain linear regression is fragile when data are scarce.** On average it
  finishes third, close behind the forest, but it has the worst single fold of
  any candidate: an MAE of 0.0750 in fold 1, when it had only 12 months (360
  stock-months) to learn from. With 12 correlated features and few training
  rows, OLS chases noise. Ridge is the same model with a penalty on large
  coefficients, and in that fold the penalty cuts the error to 0.0698. Once the
  training window grows, the two are almost indistinguishable.
- **The first two folds are bad for everything.** Together they cover
  February 2015 to March 2016, with the August 2015 volatility spike and the
  early-2016 sell-off. In fold 1, trained on only the 12 months before, every
  fitted model lost to both free baselines. That is what regime change (and
  too little history) does to a model, and it is why risk models are
  monitored rather than trusted.

Compare the two tasks honestly:

| | Direction | Volatility |
|---|---|---|
| Correlation of best single feature with target | 0.10 | 0.41 |
| Out-of-sample AUC / baseline improvement | ~0.46–0.53 (none) | ~9–10% better MAE than persistence |
| Beats the free alternative? | No | Yes, modestly: ridge in 4 of 5 folds, the forest in 3 of 5 |

Not "one fails and one triumphs" — one has no signal at all, and the other
has a modest one you can actually use.

### Where does this actually get used?

Volatility forecasts are not an academic exercise. They feed:

- **Value at Risk and Expected Shortfall** — the regulatory capital numbers
  from Notebook 02
- **Position sizing and volatility targeting** — scaling exposure down when
  risk rises
- **Option pricing and hedging** — the volatility input is the whole game
- **Margin models** — CCPs size initial margin off volatility forecasts

This is the pattern across finance. Machine learning earns its keep on
**risk, operations, fraud, credit and client analytics** — problems with
stable relationships and no adversary competing them away. It struggles on
**price direction**, where the relationships are weak by construction and
every participant is trying to find them first.

### Exercise 2 (worked solution)
Which features does the volatility model rely on? Fit the random forest
regressor on the full dataset and use `permutation_importance` (as in
Notebook 04) to rank them. Is the VIX pulling its weight, or is a stock's own
trailing volatility doing all the work?

#### Background recap: permutation importance

Notebook 04 introduced the idea. Fit a model, score it, then **shuffle one
feature's column** so its link with the target is destroyed while its
distribution is unchanged, and score again. The drop in performance is that
feature's importance. Repeat the shuffle several times (`n_repeats`) and
average, because one shuffle is noisy. Here the score is
`"neg_mean_absolute_error"` (sklearn wants "higher is better", so MAE is
negated), which makes each importance the **rise in MAE**, in volatility
points, when the feature is scrambled.

Two caveats matter for this exercise:

- **In-sample importance is optimistic.** The exercise asks for a fit on the
  full dataset; importance measured on the same rows the model learned from
  shows what the model *uses*, including anything it has memorised. So the
  solution also repeats the measurement the honest way: train on the first 40
  months and measure importance on the final, unseen 7-month block. `clone()`
  from `sklearn.base` makes a fresh unfitted copy so the full-data model is
  not overwritten.
- **Correlated features share credit.** If two features carry the same
  information (`volatility_21d` and `volatility_63d`), shuffling one costs
  little because the model can lean on the other, so both can look less
  important than the information they jointly carry.

The solution also adds up the importance of the three VIX features and of the
two own-volatility features as shares of the total, to answer the question
directly.

In [ ]:
# permutation_importance shuffles one feature at a time and measures how much the score drops
from sklearn.inspection import permutation_importance

# The same random forest regressor as in the volatility comparison, wrapped in the same pipeline
volatility_forest = Pipeline(
    [  # the two pipeline steps, in order
        ("scale", StandardScaler()),  # z-scores; harmless for trees but keeps the set-up identical
        ("model", RandomForestRegressor(  # same settings as the Task B forest
            n_estimators=300,  # number of trees averaged together
            min_samples_leaf=10,  # each leaf averages at least 10 stock-months
            random_state=RANDOM_STATE,  # reproducible bootstrap samples
            n_jobs=-1,  # use every CPU core
        )),
    ]
)

# (1) As the exercise asks: fit on the full dataset and measure importance on that same data
volatility_forest.fit(volatility_data[FEATURES], volatility_data["next_month_volatility"])
# n_repeats=10 shuffles each column ten times and averages; scoring uses MAE (negated so higher is better)
importance_full = permutation_importance(
    volatility_forest,  # the fitted pipeline
    volatility_data[FEATURES],  # features to shuffle one at a time
    volatility_data["next_month_volatility"],  # the true target
    n_repeats=10,  # ten shuffles per feature
    random_state=RANDOM_STATE,  # reproducible shuffles
    scoring="neg_mean_absolute_error",  # importance = rise in MAE when the feature is scrambled
    n_jobs=-1,  # parallelise the shuffles
)

# (2) An honest cross-check: train on the first 40 months only, measure importance on the last 7
last_train_index, last_test_index = list(splitter.split(months))[-1]
# Rows in the final fold's training window
train_ex2 = volatility_data[volatility_data["month"].isin(months[last_train_index])]
# Rows in the final, unseen test block (2017-06 to 2017-12)
test_ex2 = volatility_data[volatility_data["month"].isin(months[last_test_index])]
# sklearn's clone() makes a fresh, unfitted copy, so the full-data model above is not overwritten
from sklearn.base import clone
# Fit the copy on the training window only
holdout_forest = clone(volatility_forest).fit(train_ex2[FEATURES], train_ex2["next_month_volatility"])
# Same permutation procedure, but on data the model has never seen
importance_holdout = permutation_importance(
    holdout_forest,  # model fitted on the first 40 months
    test_ex2[FEATURES],  # the final 7 months
    test_ex2["next_month_volatility"],  # their true volatility
    n_repeats=10,  # ten shuffles per feature
    random_state=RANDOM_STATE,  # reproducible shuffles
    scoring="neg_mean_absolute_error",  # same metric as above
    n_jobs=-1,  # parallelise
)

# Collect both rankings in one table; importances_mean is the average rise in MAE per feature
volatility_importance = pd.DataFrame(
    {  # column name -> values, one value per feature
        "full_data": importance_full.importances_mean,  # in-sample importance
        "full_data_std": importance_full.importances_std,  # spread across the ten shuffles
        "held_out_last_fold": importance_holdout.importances_mean,  # out-of-sample importance
    },
    index=FEATURES,  # one row per feature name
).sort_values("full_data", ascending=False)  # most important first

# Share of total in-sample importance carried by the three VIX features
vix_features = ["vix_close", "vix_change_5d", "vix_vs_63d_avg"]
# Share carried by the stock's own trailing volatility
own_vol_features = ["volatility_21d", "volatility_63d"]
# Clip negatives to zero before taking shares, since a negative importance is just noise
positive = volatility_importance["full_data"].clip(lower=0)
# Print each group's share of the total
print(f"Share of in-sample importance from the VIX features:      {positive[vix_features].sum() / positive.sum():.1%}")
# Same calculation for the two own-volatility features
print(f"Share of in-sample importance from own trailing volatility: {positive[own_vol_features].sum() / positive.sum():.1%}")
# Show the table; values are in annualised volatility points of MAE
volatility_importance.round(4)

**Discussion.** In the full-data fit the single most important feature is the
stock's own `volatility_63d` (an MAE rise of about 0.0128 when it is
shuffled), followed by three VIX measures: `vix_vs_63d_avg` (about 0.0080),
`vix_close` (about 0.0064) and `vix_change_5d` (about 0.0047), with
`volatility_21d` next at about 0.0036. Taken together the three VIX features
carry about **41%** of the in-sample importance and the stock's own trailing
volatility about **36%**; the momentum, trend and volume features matter
little. So, in-sample, the VIX is very much pulling its weight: no single VIX
measure beats `volatility_63d`, but as a group they edge out own volatility.

The held-out check tempers that. Trained on the first 40 months and scored on
the final 7, `volatility_63d` is again the clear leader (about 0.0042),
`vix_vs_63d_avg` is second (about 0.0020), `momentum_12m` is a surprising
third (about 0.0013, which on a single 7-month block is as likely to be noise
as signal), and every other feature, including `vix_close` and
`vix_change_5d`, is close to zero. Part of the VIX's in-sample importance is
memorisation: the VIX values are identical for all 30 stocks in a month, so in
a full-data fit they act almost like a label for "which month is this", and
the forest can learn each month's average volatility from them. Out of sample,
a stock's own trailing volatility does most of the work, with the VIX's
position relative to its recent average adding a market-wide regime signal on
top. That is the economically sensible answer: risk is persistent at the stock
level and moves with market-wide fear. Note that `volatility_21d` scores low
(slightly below zero on the held-out block) partly because `volatility_63d`
already contains most of the same information.

### Exercise 3 (worked solution)
Build a third target of your own and evaluate it the same way — with a named
naive baseline that the model must beat. Suggestions:

- next month's **trading volume** (also persistent)
- whether next month's return is in the worst decile (a crash-risk classifier;
  note the severe class imbalance and use the metrics from Notebook 04)
- next month's **absolute** return, ignoring sign

State your baseline before you fit anything.

#### Solution approach: next month's absolute return

The solution takes the third suggestion, because it can be built from columns
already in the table: the target is $|r_{t+1}|$, the **size** of next
month's move ignoring its sign (`Series.abs()`). It sits between the two tasks
above: its sign-free nature makes it a cousin of volatility, but it is a
*single* draw from next month's distribution, so it is much noisier than
volatility measured from 21 daily returns.

**Baselines, stated before anything is fitted:**

1. **Last month's |return| (persistence):** forecast $|r_{t+1}| = |r_t|$, using
   `momentum_1m`, which is this month's return.
2. **Train-set mean:** forecast the average absolute return in the training
   window for every stock.
3. **Volatility rule:** if monthly returns were normal with standard deviation
   $\sigma_m$, then $E|r| = \sigma_m\sqrt{2/\pi}$. With $\sigma_m$ =
   `volatility_21d` $/\sqrt{12}$, this is a free forecast from Task B's best
   single input.

The models are ridge (`alpha=10`) and the random forest regressor with the
same settings as Task B, evaluated on the same five walk-forward folds with
MAE and $R^2$. `clone(estimator)` gives each fold a fresh copy of the model.
The cell ends by comparing the forest with whichever baseline scored best,
fold by fold.

In [ ]:
# New target: next month's ABSOLUTE return, |r|, the size of the move regardless of sign
# clone() makes a fresh, unfitted copy of an estimator (also imported in Exercise 2; repeated so this cell stands alone)
from sklearn.base import clone
# Drop any row without a next-month return; .copy() gives an independent table we can add a column to
abs_data = data.dropna(subset=["next_month_return"]).copy()
# .abs() takes the absolute value of every element of the column
abs_data["next_month_abs_return"] = abs_data["next_month_return"].abs()

# Baselines are named BEFORE anything is fitted:
#   1. "Last month's |return|": persistence, predict next |r| = this month's |r| (momentum_1m is this month's return)
#   2. "Train-set mean": predict the average |r| seen in the training window, the same for every row
#   3. "Volatility rule": a monthly move is roughly annual vol / sqrt(12); for a normal variable E|r| = sigma * sqrt(2/pi)
abs_models = {
    "Last month's |return| (no model)": "persistence",  # baseline 1; string flags mark the free baselines
    "Train-set mean (no model)": "constant",  # baseline 2
    "Volatility rule (no model)": "vol_rule",  # baseline 3
    "Ridge regression": Ridge(alpha=10.0),  # same penalty as Task B
    "Random forest": RandomForestRegressor(  # ensemble of regression trees
        n_estimators=300, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1  # same settings as Task B
    ),
}

# One row per (model, fold); a new name so earlier results are not overwritten
abs_rows = []
# Outer loop over the five candidates
for name, estimator in abs_models.items():
    # Inner loop over the same walk-forward folds
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        # Rows in this fold's training window
        train_ex3 = abs_data[abs_data["month"].isin(months[train_index])]
        # Rows in the following test block
        test_ex3 = abs_data[abs_data["month"].isin(months[test_index])]
        # The truth we are scoring against, as a NumPy array
        y_true_ex3 = test_ex3["next_month_abs_return"].to_numpy()
        # Persistence: this month's absolute return is the forecast
        if estimator == "persistence":
            # momentum_1m is this month's return; abs() drops the sign
            y_pred_ex3 = test_ex3["momentum_1m"].abs().to_numpy()
        # Constant: the training-window mean, repeated len(test) times with np.full
        elif estimator == "constant":
            # Average |r| over the training window, copied once per test row
            y_pred_ex3 = np.full(len(test_ex3), train_ex3["next_month_abs_return"].mean())
        # Volatility rule: 21-day annualised vol scaled to one month, times sqrt(2/pi)
        elif estimator == "vol_rule":
            # Annual vol / sqrt(12) = monthly sigma; times sqrt(2/pi) = expected absolute size of a normal move
            y_pred_ex3 = (test_ex3["volatility_21d"] / np.sqrt(12) * np.sqrt(2 / np.pi)).to_numpy()
        # Otherwise fit a real model inside the scaler pipeline, on training rows only
        else:
            # Scaler plus a fresh copy of the model, fitted on this fold's training rows
            fitted_ex3 = Pipeline(
                [("scale", StandardScaler()), ("model", clone(estimator))]  # clone gives a fresh model each fold
            ).fit(train_ex3[FEATURES], train_ex3["next_month_abs_return"])  # learn from the past only
            # Predict |r| for the unseen test rows
            y_pred_ex3 = fitted_ex3.predict(test_ex3[FEATURES])
        # Score this fold with the same two metrics as Task B
        abs_rows.append(
            {  # one record per model and fold
                "model": name,  # which candidate
                "fold": fold,  # which fold
                "mae": mean_absolute_error(y_true_ex3, y_pred_ex3),  # average miss, in return units
                "r2": r2_score(y_true_ex3, y_pred_ex3),  # out-of-sample R squared
            }
        )

# Collect into a DataFrame
abs_results = pd.DataFrame(abs_rows)
# Average across folds and sort so the best (lowest MAE) is on top
abs_summary = abs_results.groupby("model").agg(
    mean_mae=("mae", "mean"),  # average error
    worst_fold_mae=("mae", "max"),  # the bad-regime fold
    mean_r2=("r2", "mean"),  # can be negative out of sample
).sort_values("mean_mae")  # best (lowest error) first

# Per-fold MAE, models as rows, folds as columns
abs_per_fold = abs_results.pivot(index="model", columns="fold", values="mae")
# How often does the forest beat the best free baseline, fold by fold?
best_baseline = abs_summary[abs_summary.index.str.contains("no model")].index[0]
# Count folds where the forest's MAE is lower than the best baseline's
abs_wins = int((abs_per_fold.loc["Random forest"] < abs_per_fold.loc[best_baseline]).sum())
# Report the comparison against the named baseline, as the recap asks
print(f"Best free baseline: {best_baseline}")
# Win count and relative MAE improvement of the forest over that baseline
print(f"Random forest beats it in {abs_wins} of {abs_per_fold.shape[1]} folds; "
      f"mean MAE improvement {1 - abs_per_fold.loc['Random forest'].mean() / abs_per_fold.loc[best_baseline].mean():.1%}")  # positive = forest better
# Display the summary table
abs_summary.round(4)

**Discussion.** The best free baseline is the **train-set mean** (mean MAE
about 0.0269, i.e. a typical miss of 2.7 percentage points of monthly
return). The random forest beats it in **4 of 5 folds**, but only by about
**1.8%** in MAE (0.0264), and its out-of-sample $R^2$ is about -0.05 —
slightly *worse* than the train-set mean's -0.03. The two metrics can
disagree because $R^2$ works on squared errors, which weight the occasional
large miss heavily, while MAE weights every miss in proportion to its size.
Ridge (0.0271) does not beat the constant on average at all. An edge this
small, which changes sign depending on the metric you read, is barely
distinguishable from no edge.

Two lessons. First, persistence of a single month's absolute return is a
terrible forecast (MAE about 0.0378, $R^2$ about -1.13): one month's move is a
noisy sample, unlike volatility built from 21 daily returns. Even the
volatility rule (0.0278) loses to the plain average, because one monthly draw
scatters widely around its expected size. Second, the edge here is far
smaller than Task B's 9–10%, for the same reason: the *expected* size of next
month's move is somewhat predictable, but the realised size is mostly noise.
Choosing a target that measures what you can actually forecast (volatility
over many days rather than one monthly return) matters as much as choosing
the model.

---

## Recap

1. Always name the naive baseline first: the base rate for classification,
   persistence for a time series. Report the improvement over it, not the raw
   score.
2. A classifier that has learned the base rate looks accurate and knows
   nothing. Check what share of predictions are in one class.
3. AUC near 0.5 means no signal, whatever the accuracy says.
4. Put a standard error on any claimed improvement before believing it.
5. Direction is hard because it is competed away. Risk is easier because it
   is not — but "easier" here means a 9–10% error reduction, not a crystal
   ball.
6. Out-of-sample $R^2$ can be negative and is unstable under regime shifts.
   Pick the metric that answers the question you actually asked.
7. Simple, regularised models do about as well as complex ones: ridge
   edged out the random forest, and its penalty rescued plain linear
   regression when training data were scarce (fold 1).
8. Reporting a clean null result is a professional skill.

---

## Where to take this next

Ideas for a term project, roughly in order of difficulty:

- Add fundamentals (P/E, leverage, profitability) and retest direction.
- Replace the point forecast of volatility with a GARCH(1,1) and compare.
- Build the crash-risk classifier from Exercise 3 and set its threshold from
  an explicit cost of being wrong, as in Notebook 04.
- Take the credit model from Notebook 04 and write the model card: intended
  use, training data, fair-lending testing, monitoring plan.
- Estimate what trading costs would do to the Task A backtest, and find the
  cost level at which the strategy breaks even.

---

## Your turn

You have now run every cell, read the explanations and worked through the
three exercise solutions. Next:

1. Make sure the notebook ran from top to bottom without errors (**Run All**),
   so its outputs are saved in the file.
2. Complete the **Canvas comprehension quiz for Notebook 05**. It asks about
   what specific lines of code do, the methods in the background sections
   (walk-forward validation, baselines, ridge, MAE and $R^2$, volatility
   clustering, market efficiency), and the results you saw.
3. Commit and push your run notebook:

```
git add notebooks/05_ml_market_prediction.ipynb
git commit -m "Ran notebook 05"
git push
```

Next: this is the last notebook in Lab 3. The term-project ideas above are a
good place to start if you want to take any of it further.